# Web Server Log Analysis
Analyzing the NASA HTTP access log dataset to identify suspicious network activity.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3

## Load Data

In [ ]:
# Load the log file
df = pd.read_csv('../data/access_logs.csv')
df.head()

## Explore Data

In [ ]:
# Basic info
print(df.shape)
print(df.dtypes)
df.describe()

## SQLITE

In [ ]:
conn = sqlite3.connect('logs_database.db')
conn.row_factory = sqlite3.Row

df.to_sql('logs', conn, if_exists='replace', index=False)

In [ ]:
# Check column types from the schema
rows = conn.execute("PRAGMA table_info(logs)").fetchall()
for row in rows:
    print(row[0], row[1], row[2], row[3], row[4])

In [ ]:
count_query = "SELECT COUNT(*) FROM logs"
count_df = pd.read_sql_query(count_query, conn)
print(count_df)

In [ ]:
cursor = conn.execute('SELECT * FROM logs LIMIT 1')
col_names = [description[0] for description in cursor.description]
print(col_names)

In [ ]:
# finding ipaddresses with the most requests
rows = conn.execute(""" 
                    SELECT host, COUNT(*) AS requests_count
                    FROM logs
                    GROUP BY host
                    ORDER BY requests_count DESC
                    LIMIT 10
                    """).fetchall()

for row in rows:
    print(f"{row['host']}: {row['requests_count']}")

In [ ]:
#seeing how many 404 errors occurred
rows = conn.execute(""" 
                    SELECT host, status, COUNT(*) AS total_404s
                    FROM logs
                    WHERE status = ?
                    GROUP BY host
                    ORDER BY total_404s DESC
                    """, (404,)).fetchall()

for row in rows:
    print(f"{row['host']}; {row['total_404s']}")

In [ ]:
#seeing how many 500 errors occurred
rows = conn.execute(""" 
                    SELECT status, COUNT(*) AS total_500
                    FROM logs
                    WHERE status = ?
                    """, (500,)).fetchall()

for row in rows:
    print(f"Total 500 status: {row['total_500']}")

In [ ]:
#most requested urls
rows = conn.execute(""" 
                    SELECT request, COUNT(*) AS total_requests
                    FROM logs
                    GROUP BY request
                    ORDER BY total_requests DESC
                    """).fetchall()

for row in rows:
    print(f"{row['request']}; {row['total_requests']}")

In [ ]:
#converted timestamp column to datetime data type
from datetime import datetime

rows = conn.execute(""" 
                   SELECT rowid, timestamp
                   FROM logs
                   """).fetchall()

for row in rows:
    dt = datetime.strptime(row["timestamp"], "%d/%b/%Y:%H:%M:%S %z")
    iso = dt.strftime("%Y-%m-%d %H:%M:%S")
    
    conn.execute("""
                 UPDATE logs SET timestamp = ?
                 WHERE rowid = ?
                 """, (iso, row["rowid"]))
conn.commit()

In [ ]:
#unusual hours 12amm - 5am EASTERN

off_hours = conn.execute("""
                    SELECT host, COUNT(*) AS request_count 
                    FROM logs
                    WHERE timestamp BETWEEN ? AND ?
                    GROUP BY host
                    ORDER BY request_count DESC
                    """, ("1995-07-01 00:00:00", "1995-07-01 05:00:00")).fetchall()

for row in off_hours:
    print(row['host'], row['request_count'])

In [ ]:
#unusual hours 12amm - 5am EASTERN

work_hours = conn.execute("""
                    SELECT host, COUNT(*) AS request_count 
                    FROM logs
                    WHERE timestamp BETWEEN ? AND ?
                    GROUP BY host
                    ORDER BY request_count DESC
                    """, ("1995-07-01 09:00:00", "1995-07-01 17:00:00")).fetchall()

for row in work_hours:
    print(row['host'], row['request_count'])

In [ ]:
#getting union of ip between workday and off hours
#sending it to pandas df
ip_traffic_df = pd.read_sql_query("""
                        SELECT host, COUNT(*) AS request_count, 'overnight' AS WINDOW
                        FROM logs
                        WHERE timestamp BETWEEN ? AND ?
                        GROUP BY host
                        
                        UNION ALL
                        
                        SELECT host, COUNT(*) AS request_count, 'business' AS window
                        FROM logs
                        WHERE timestamp BETWEEN ? AND ?
                        GROUP BY host
                        
                        ORDER BY host, window
                        """, conn, params = ("1995-07-01 00:00:00", "1995-07-01 05:00:00",
                            "1995-07-01 09:00:00", "1995-07-01 17:00:00"))
ip_traffic_df.head()

In [ ]:
traffic_df = pd.read_sql_query("""
                        SELECT host, COUNT(*) AS request_count,
                        CASE
                            WHEN CAST(strftime('%H', timestamp) AS INTEGER) BETWEEN 0 AND 5 THEN 'overnight'
                            WHEN CAST(strftime('%H', timestamp) AS INTEGER) BETWEEN 9 AND 17 THEN 'business'
                            ELSE 'other'
                        END AS window
                        FROM logs    
                        GROUP BY host, window                    
                        ORDER BY host, window
                        """, conn)
traffic_df.head()

In [ ]:
#looking at sus ip traffic in pandas df
traffic_df = traffic_df.pivot(index='host', 
                                          columns='window', 
                                          values='request_count')


traffic_df = traffic_df.fillna(0)
sus = traffic_df[(traffic_df['overnight'] > 50) & (traffic_df['business'] == 0)]
sus = sus.sort_values(by='overnight', ascending=False)

sus.head(20) 

In [ ]:
rows = conn.execute(""" 
                    SELECT host, request, COUNT(*) as count
                    FROM logs
                    WHERE host = '202.239.48.3'
                    GROUP BY request
                    ORDER BY count DESC
                    LIMIT 10
                    """)

for row in rows:
    print(row['request'])

In [ ]:
rows = conn.execute(""" 
                    SELECT host, request, COUNT(*) as count
                    FROM logs
                    WHERE host = 'ppp120.amaranth.com'
                    GROUP BY request
                    ORDER BY count DESC
                    LIMIT 10
                    """)

for row in rows:
    print(row['request'])

In [ ]:
rows = conn.execute(""" 
                    SELECT host, request, COUNT(*) as count
                    FROM logs
                    WHERE host = 'mnssec3.dukepower.com'
                    GROUP BY request
                    ORDER BY count DESC
                    LIMIT 10
                    """)

for row in rows:
    print(row['request'])

In [ ]:
rows = conn.execute(""" 
                    SELECT host, request, COUNT(*) as count
                    FROM logs
                    WHERE host = 'xsi12.komaba.ecc.u-tokyo.ac.jp'
                    GROUP BY request
                    ORDER BY count DESC
                    LIMIT 10
                    """)

for row in rows:
    print(row['request'])

In [ ]:
rows = conn.execute(""" 
                    SELECT host, request, COUNT(*) as count
                    FROM logs
                    WHERE host = 'jbiagioni.npt.nuwc.navy.mil'
                    GROUP BY request
                    ORDER BY count DESC
                    LIMIT 10
                    """)

for row in rows:
    print(row['request'])

In [ ]:
rows = conn.execute("""
                    SELECT host, request, COUNT(*) as count
                    FROM logs 
                    WHERE request LIKE '%admin%'
                    OR request LIKE '%passwd%'
                    OR request LIKE '%cgi-bin%'
                    GROUP BY host, request
                    ORDER BY count DESC
                    LIMIT 20
                    """)

for row in rows:
    print(row['host'], row['request'], row['count'])

In [ ]:
rows = conn.execute(""" 
                    SELECT host, request, COUNT(*) as count
                    FROM logs
                    WHERE host = 163.205.1.45
                    GROUP BY request
                    ORDER BY count DESC
                    LIMIT 10
                    """)

for row in rows:
    print(row['request'])

In [ ]:
conn.close()